# Notebook 2: Fixed Menstruation Pattern Classification & Permutation Test
## PhD Research — Probabilistic and Statistical Analysis of the Menstrual Cycle from a Halachic Perspective
**Author:** Dvir Ross, Shenkar College

This notebook covers all six types of fixed (קבועה) menstruation defined in Halachic law,
their frequency in the dataset, and a permutation-based hypothesis test showing
that the observed pattern frequencies are non-random.

### Six Halachic Fixed Menstruation Types
| # | Name (Hebrew) | Name (English) | Definition |
|---|--------------|----------------|-----------|
| 1 | הפלגה (Haflaga) | Interval-fixed | 3 consecutive equal cycle lengths |
| 2 | דילוג (Dilug) | Arithmetic-progression | Constant non-zero difference between consecutive cycles |
| 3 | השבוע (Week) | Weekly | Two consecutive cycles at a length ≡ 0 (mod 7) from a base |
| 4 | השבוע בדילוג (Week-Dilug) | Weekly-specific | Two consecutive cycles at length 30 |
| 5 | דילוג בתוך דילוג (Dilug-in-Dilug) | Second-order AP | Differences themselves form an AP |
| 6 | חוזר חלילה (Recurring) | Cyclic triplet | A triplet of cycle lengths repeats exactly |


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## 1. Load and Prepare Data

In [ ]:
data = pd.read_csv('datasets/FilteredData.csv')
df = data[['ClientID','CycleNumber','LengthofCycle']].copy()
df['LengthofCycle'] = df['LengthofCycle'] + 1   # Halachic +1 convention

# Build partition index: (start, end) per client
change_index = df.index[df['ClientID'] != df['ClientID'].shift()].tolist()
change_index.append(len(df))
change_indexes = [(change_index[i-1], change_index[i]) for i in range(1, len(change_index))]

print(f"Clients: {df['ClientID'].nunique()}")
print(f"Total cycles: {len(df)}")
print(f"Cycle length range: {df['LengthofCycle'].min()} – {df['LengthofCycle'].max()} days")

## 2. Pattern Detection Functions

In [ ]:
# ── Type 1: Haflaga ──────────────────────────────────────────────────────
def haflaga_count(lengths):
    """Three consecutive identical cycle lengths."""
    n = 0
    for s, e in change_indexes:
        i = s + 2
        while i < e:
            if lengths[i] == lengths[i-1] == lengths[i-2]:
                n += 1
                while i + 1 < e and lengths[i+1] == lengths[i]:
                    i += 1
            i += 1
    return n

# ── Type 2: Dilug ─────────────────────────────────────────────────────────
def dilug_count(lengths):
    """Constant non-zero difference: L[i]-L[i-1] == L[i-1]-L[i-2] != 0."""
    n = 0
    for s, e in change_indexes:
        i = s + 2
        while i < e:
            d = lengths[i] - lengths[i-1]
            if d != 0 and d == lengths[i-1] - lengths[i-2]:
                n += 1
                while i + 1 < e and lengths[i+1] - lengths[i] == d:
                    i += 1
            i += 1
    return n

# ── Type 3: Week (generic) ────────────────────────────────────────────────
def week_count(lengths):
    """Two consecutive cycles at same length, that length is ≡ 3 (mod 7) above min."""
    weekday_lengths = [v for v in range(df['LengthofCycle'].min() + 3,
                                        df['LengthofCycle'].max() + 1, 7)]
    n = 0
    for s, e in change_indexes:
        i = s + 1
        while i < e:
            if lengths[i] in weekday_lengths and lengths[i] == lengths[i-1]:
                n += 1
                while i + 1 < e and lengths[i+1] == lengths[i]:
                    i += 1
            i += 1
    return n

# ── Type 4: Week-Dilug (length 30 specifically) ───────────────────────────
def week_dilug_count(lengths):
    """Two consecutive cycles both at length 30."""
    n = 0
    for s, e in change_indexes:
        i = s + 1
        while i < e:
            if lengths[i] == 30 and lengths[i-1] == 30:
                n += 1
                while i + 1 < e and lengths[i+1] == 30:
                    i += 1
            i += 1
    return n

# ── Type 5: Dilug-in-Dilug ───────────────────────────────────────────────
def dilug_in_dilug_count(lengths):
    """Second-order AP: differences b_i = L[i]-L[i-1] themselves form an AP."""
    n = 0
    for s, e in change_indexes:
        i = s + 3
        while i < e:
            b1 = lengths[i-2] - lengths[i-3]
            b2 = lengths[i-1] - lengths[i-2]
            b3 = lengths[i]   - lengths[i-1]
            d = b2 - b1
            if d != 0 and b3 - b2 == d:
                n += 1
                b, dd = b3, d
                while i + 1 < e and lengths[i+1] == lengths[i] + b + dd:
                    i += 1
                    b += dd
            i += 1
    return n

# ── Type 6: Recurring (Chozer Chalila) ────────────────────────────────────
def recurring_count(lengths):
    """A triplet (L[i], L[i+1], L[i+2]) that previously appeared as three
       consecutive values for the same client — the pattern recurs exactly."""
    n = 0
    for s, e in change_indexes:
        seen_triplets = set()
        # Seed with the first triplet
        if e - s >= 3:
            seen_triplets.add((lengths[s], lengths[s+1], lengths[s+2]))
        i = s + 3
        while i < e:
            triplet = (lengths[i-2], lengths[i-1], lengths[i])
            if triplet in seen_triplets:
                n += 1
                i += 3  # advance past the recurring triplet
                continue
            seen_triplets.add(triplet)
            i += 1
    return n

print("Pattern detection functions defined.")

## 3. Observed Frequencies

In [ ]:
lengths = df['LengthofCycle'].copy()

pattern_names = ['Haflaga', 'Dilug', 'Week', 'Week-Dilug', 'Dilug-in-Dilug', 'Recurring']
count_fns = [haflaga_count, dilug_count, week_count,
             week_dilug_count, dilug_in_dilug_count, recurring_count]

observed = np.array([fn(lengths) for fn in count_fns])

print("Observed fixed-menstruation pattern frequencies:")
print("-" * 45)
for name, cnt in zip(pattern_names, observed):
    print(f"  {name:<20}: {cnt:>4}   ({cnt/len(df)*100:.2f}% of cycles)")
print(f"  {'Total cycles':<20}: {len(df):>4}")

## 4. Joint Distribution — Clients with Multiple Pattern Types

In [ ]:
# For each client, which patterns did they ever exhibit?
client_patterns = {name: set() for name in pattern_names}

# We need per-client detection; build a helper that restricts change_indexes
def per_client_count(fn, client_id):
    ci = df[df['ClientID'] == client_id]
    if len(ci) < 3:
        return 0
    # Temporarily re-index
    sub = ci['LengthofCycle'].reset_index(drop=True)
    old_ci = change_indexes
    dummy_ci = [(0, len(sub))]
    # patch change_indexes globally for the call
    global change_indexes
    change_indexes = dummy_ci
    result = fn(sub)
    change_indexes = old_ci
    return result

pattern_matrix = []
for cid in df['ClientID'].unique():
    row = {'ClientID': cid}
    for name, fn in zip(pattern_names, count_fns):
        row[name] = 1 if per_client_count(fn, cid) > 0 else 0
    pattern_matrix.append(row)

pm = pd.DataFrame(pattern_matrix).set_index('ClientID')
pm['Total_types'] = pm.sum(axis=1)

print("Clients by number of pattern types observed:")
print(pm['Total_types'].value_counts().sort_index().to_string())
print(f"\nClients with at least one fixed pattern: {(pm['Total_types'] > 0).sum()}")
print("\nPattern prevalence (clients):")
print(pm[pattern_names].sum().to_string())

## 5. Permutation Test

**Null hypothesis H₀:** The observed vector of pattern frequencies is no different from
what would be obtained if cycle lengths were randomly rearranged (preserving the
longitudinal structure of the dataset).

**Test statistic:** Euclidean distance from the mean simulated frequency vector.

**Method:** 10,000 permutations; p-value = proportion of permutations whose
test statistic exceeds the observed statistic.

Reference: Ross (2023), vector-of-statistics permutation test framework.

In [ ]:
np.random.seed(17)
NUM_PERMUTATIONS = 10_000

simulations = []
for _ in range(NUM_PERMUTATIONS):
    perm_lengths = np.random.permutation(lengths)
    sim_row = [fn(perm_lengths) for fn in count_fns]
    simulations.append(sim_row)

sim_df = pd.DataFrame(simulations, columns=pattern_names)
mean_freq = sim_df.mean().values

print("Permutation simulation summary (mean ± SD):")
print("-" * 45)
for i, name in enumerate(pattern_names):
    print(f"  {name:<20}: {mean_freq[i]:.2f} ± {sim_df.iloc[:,i].std():.2f}")

In [ ]:
# ── Compute test statistic ───────────────────────────────────────────────
dist_observed = np.linalg.norm(observed - mean_freq)
dist_simulated = np.linalg.norm(
    sim_df.values - mean_freq, axis=1
)

p_value = np.mean(dist_simulated >= dist_observed)
percentile = np.searchsorted(np.sort(dist_simulated), dist_observed, side='right') / NUM_PERMUTATIONS

print("=" * 55)
print("  PERMUTATION TEST RESULTS")
print("=" * 55)
print(f"  Observed frequency vector:  {observed}")
print(f"  Mean simulated vector:      {mean_freq.round(1)}")
print(f"  Observed distance from mean:{dist_observed:.4f}")
print(f"  p-value (permutation):      {p_value:.4f}")
print(f"  Percentile:                 {percentile:.4f}")
print(f"  Conclusion: {'H₀ REJECTED' if p_value < 0.05 else 'H₀ not rejected'} at α = 0.05")

## 6. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── (a) Observed vs. expected frequencies ─────────────────────────────────
ax = axes[0]
x = np.arange(len(pattern_names))
w = 0.35
ax.bar(x - w/2, observed, w, label='Observed', color='steelblue', alpha=0.85)
ax.bar(x + w/2, mean_freq, w, label='Expected (null)', color='tomato', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(pattern_names, rotation=25, ha='right')
ax.set_ylabel('Frequency')
ax.set_title('(a) Observed vs. Expected Pattern Frequencies')
ax.legend()

# ── (b) Null distribution of test statistic ──────────────────────────────
ax = axes[1]
kde = stats.gaussian_kde(dist_simulated)
xs = np.linspace(dist_simulated.min(), dist_simulated.max(), 300)
ax.plot(xs, kde(xs), color='gray', lw=2)
ax.fill_between(xs, kde(xs), where=(xs >= dist_observed),
                color='tomato', alpha=0.4, label=f'p = {p_value:.4f}')
ax.axvline(dist_observed, color='crimson', ls='--', lw=2,
           label=f'Observed distance = {dist_observed:.2f}')
ax.set_xlabel('Distance from Mean Frequency Vector')
ax.set_ylabel('Density')
ax.set_title('(b) Null Distribution of Test Statistic')
ax.legend()

plt.tight_layout()
plt.savefig('figures/fig2_permutation_test.png', bbox_inches='tight')
plt.show()
print("Figure saved to figures/fig2_permutation_test.png")

## 7. Summary Table (for paper)

In [ ]:
summary = pd.DataFrame({
    'Pattern': pattern_names,
    'Observed_Count': observed,
    'Simulated_Mean': mean_freq.round(2),
    'Simulated_SD': sim_df.std().round(2).values,
    'Z_score': ((observed - mean_freq) / sim_df.std().values).round(3),
    'Pct_of_Cycles': (observed / len(df) * 100).round(2),
})
print("Summary table:")
print(summary.to_string(index=False))
print(f"\nOverall permutation test: distance = {dist_observed:.3f}, p = {p_value:.4f}")